In [1]:
# Cell 1: Install dependencies
import subprocess, sys

pkgs = [
    'wfdb', 'torch', 'numpy', 'scipy', 'matplotlib',
    'opencv-python-headless', 'Pillow', 'gradio>=5.0.0', 'nbconvert',
]

for p in pkgs:
    print(f'Installing {p} ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=True)
print('All dependencies installed.')


Installing wfdb ...


Installing torch ...


Installing numpy ...


Installing scipy ...


Installing matplotlib ...


Installing opencv-python-headless ...


Installing Pillow ...


Installing gradio>=5.0.0 ...


Installing nbconvert ...


All dependencies installed.


In [2]:
# Cell 2: Download 2 real clinical 12-lead ECG records from PhysioNet PTB-XL
import os, wfdb
import numpy as np

os.makedirs('sample_ecgs', exist_ok=True)

PTBXL_DB = 'ptb-xl'

rec_normal = '00001_lr'
for ext in ['.hea', '.dat']:
    wfdb.dl_files(PTBXL_DB, 'sample_ecgs', files=['records100/00000/' + rec_normal + ext], keep_subdirs=False)
sig_n, fields_n = wfdb.rdsamp('sample_ecgs/' + rec_normal)
print(f'Normal SR record loaded: shape={sig_n.shape}, fs={fields_n["fs"]}')

rec_mi = '00008_lr'
for ext in ['.hea', '.dat']:
    wfdb.dl_files(PTBXL_DB, 'sample_ecgs', files=['records100/00000/' + rec_mi + ext], keep_subdirs=False)
sig_mi, fields_mi = wfdb.rdsamp('sample_ecgs/' + rec_mi)
print(f'MI alert record loaded: shape={sig_mi.shape}, fs={fields_mi["fs"]}')

assert sig_n.shape == (1000, 12), f'Expected (1000, 12), got {sig_n.shape}'
assert sig_mi.shape == (1000, 12), f'Expected (1000, 12), got {sig_mi.shape}'
print('Both records verified: (1000, 12) 12-lead ECG.')


Finished downloading files


Finished downloading files
Normal SR record loaded: shape=(1000, 12), fs=100


Finished downloading files


Finished downloading files
MI alert record loaded: shape=(1000, 12), fs=100
Both records verified: (1000, 12) 12-lead ECG.


In [3]:
# Cell 3: Bandpass filter, biomarker extractor, and ECG1DResNet classifier
import numpy as np
import torch
import torch.nn as nn
from scipy import signal

LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
SAMPLE_RATE = 500


def bandpass_filter(ecg, lowcut=0.5, highcut=40.0, fs=500, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, ecg, axis=0)


def extract_biomarkers(ecg, fs=500):
    ecg_f = bandpass_filter(ecg, fs=fs)
    lead_ii = ecg_f[:, 1]
    peaks, _ = signal.find_peaks(lead_ii, distance=int(0.4*fs), prominence=0.1)
    if len(peaks) > 1:
        rr_intervals = np.diff(peaks) / fs
        bpm = 60.0 / np.mean(rr_intervals)
        rr_irregularity = np.std(rr_intervals) / np.mean(rr_intervals) if np.mean(rr_intervals) > 0 else 0.0
    else:
        bpm = 75.0
        rr_irregularity = 0.0
    
    st_leads = [1, 5, 9, 10, 11]
    st_devs = []
    for li in st_leads:
        seg = ecg_f[int(0.3*fs):int(0.5*fs), li]
        st_devs.append(float(np.mean(seg)))
    st_deviation = float(np.mean(np.abs(st_devs)))
    
    return {
        'bpm': float(bpm),
        'rr_irregularity': float(rr_irregularity),
        'st_deviation': st_deviation,
    }


class ECG1DResNet(nn.Module):
    def __init__(self, in_channels=12, num_classes=5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.block1 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
        )
        self.shortcut = nn.Conv1d(64, 64, kernel_size=1)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes),
            nn.Sigmoid(),
        )
    
    def forward(self, x):
        x = self.stem(x)
        out = self.block1(x) + self.shortcut(x)
        out = torch.relu(out)
        return self.classifier(out)


CONDITIONS = ['Normal SR', 'Atrial Fibrillation', 'STEMI / Acute MI', 'Bradycardia', 'Tachycardia']

model = ECG1DResNet()
model.eval()
print('ECG1DResNet model initialized.')


ECG1DResNet model initialized.


In [4]:
# Cell 4: Run inference on both PTB-XL records
import time, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
from PIL import Image

SAMPLES = {
    'normal_sr': 'sample_ecgs/00001_lr',
    'mi_alert': 'sample_ecgs/00008_lr',
}

def run_inference(sig, label, fs=500):
    t0 = time.perf_counter()
    
    biomarkers = extract_biomarkers(sig, fs=fs)
    
    x = torch.from_numpy(sig.T).float().unsqueeze(0)
    with torch.no_grad():
        nn_probs = model(x).numpy()[0]
    
    triage = {
        'sample': label,
        'bpm': biomarkers['bpm'],
        'rr_irregularity': biomarkers['rr_irregularity'],
        'st_deviation': biomarkers['st_deviation'],
        'nn_probs': [float(nn_probs[0]), float(nn_probs[1]), float(nn_probs[2]), float(nn_probs[3]), float(nn_probs[4])],
        'nn_conditions': CONDITIONS,
        'latency_ms': 0.0,
        'recommendation': 'Routine follow-up'
    }
    
    latency_ms = (time.perf_counter() - t0) * 1000.0
    triage['latency_ms'] = round(latency_ms, 2)
    
    if triage['nn_probs'][2] > 0.7 or biomarkers['st_deviation'] > 0.15:
        triage['recommendation'] = 'URGENT: Possible acute MI - immediate cardiology referral'
    elif biomarkers['rr_irregularity'] > 0.12 or triage['nn_probs'][1] > 0.7:
        triage['recommendation'] = 'Urgent: Possible atrial fibrillation - ECG confirmation'
    elif biomarkers['bpm'] < 50:
        triage['recommendation'] = 'Urgent: Bradycardia - clinical review'
    elif biomarkers['bpm'] > 120:
        triage['recommendation'] = 'Urgent: Tachycardia - clinical review'
    else:
        triage['recommendation'] = 'Routine follow-up'
    
    return triage

def plot_12lead(sig, label, out_path, fs=500):
    fig, axes = plt.subplots(4, 3, figsize=(15, 10))
    fig.suptitle(f'12-Lead ECG: {label}', fontsize=14)
    t = np.arange(sig.shape[0]) / fs
    for i, ax in enumerate(axes.flat):
        ax.plot(t, sig[:, i], linewidth=0.5)
        ax.set_title(LEAD_NAMES[i], fontsize=9)
        ax.set_xlabel('s', fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f'Saved plot: {out_path}')

results = {}
for key in list(SAMPLES.keys()):
    path = SAMPLES[key]
    sig, fields = wfdb.rdsamp(path)
    t0 = time.perf_counter()
    actual_fs = int(fields['fs'])
    biomarkers = extract_biomarkers(sig, fs=actual_fs)
    
    x = torch.from_numpy(sig.T).float().unsqueeze(0)
    with torch.no_grad():
        nn_probs = model(x).numpy()[0]
    
    triage = {
        'sample': key,
        'bpm': biomarkers['bpm'],
        'rr_irregularity': biomarkers['rr_irregularity'],
        'st_deviation': biomarkers['st_deviation'],
        'nn_probs': [float(nn_probs[0]), float(nn_probs[1]), float(nn_probs[2]), float(nn_probs[3]), float(nn_probs[4])],
        'nn_conditions': CONDITIONS,
        'latency_ms': 0.0,
        'recommendation': 'Routine follow-up'
    }
    
    latency_ms = (time.perf_counter() - t0) * 1000.0
    triage['latency_ms'] = round(latency_ms, 2)
    
    if triage['nn_probs'][2] > 0.7 or biomarkers['st_deviation'] > 0.15:
        triage['recommendation'] = 'URGENT: Possible acute MI - immediate cardiology referral'
    elif biomarkers['rr_irregularity'] > 0.12 or triage['nn_probs'][1] > 0.7:
        triage['recommendation'] = 'Urgent: Possible atrial fibrillation - ECG confirmation'
    elif biomarkers['bpm'] < 50:
        triage['recommendation'] = 'Urgent: Bradycardia - clinical review'
    elif biomarkers['bpm'] > 120:
        triage['recommendation'] = 'Urgent: Tachycardia - clinical review'
    else:
        triage['recommendation'] = 'Routine follow-up'
    
    results[key] = triage
    print(json.dumps(triage, indent=2))
    
    fname = 'normal' if key == 'normal_sr' else 'mi_alert'
    out_png = 'output_' + fname + '.png'
    plot_12lead(sig, key, out_png, fs=actual_fs)

print('\nInference complete for both PTB-XL records.')


{
  "sample": "normal_sr",
  "bpm": 63.897763578274756,
  "rr_irregularity": 0.019316674278186393,
  "st_deviation": 0.013404399976027295,
  "nn_probs": [
    0.5313810706138611,
    0.5022243857383728,
    0.4966653883457184,
    0.4800623059272766,
    0.4934315085411072
  ],
  "nn_conditions": [
    "Normal SR",
    "Atrial Fibrillation",
    "STEMI / Acute MI",
    "Bradycardia",
    "Tachycardia"
  ],
  "latency_ms": 6.2,
  "recommendation": "Routine follow-up"
}


Saved plot: output_normal.png
{
  "sample": "mi_alert",
  "bpm": 73.74301675977652,
  "rr_irregularity": 0.016796979193712706,
  "st_deviation": 0.02091773774046276,
  "nn_probs": [
    0.5300569534301758,
    0.5001305937767029,
    0.49310675263404846,
    0.4810641407966614,
    0.4923461675643921
  ],
  "nn_conditions": [
    "Normal SR",
    "Atrial Fibrillation",
    "STEMI / Acute MI",
    "Bradycardia",
    "Tachycardia"
  ],
  "latency_ms": 3.09,
  "recommendation": "Routine follow-up"
}


Saved plot: output_mi_alert.png

Inference complete for both PTB-XL records.
